In [ ]:
%cd c:\Users\almei\Documents\GitHub\precision-data
from pathlib import Path
from aatm.aio.translators import AsyncGeminiTranslator
from aatm.terminology_mapper import TerminologyMapper
from aatm.retrievers import CHROMADB_RETRIEVER_MODEL_REGISTRY as model_registry
from aatm.retrievers import load_chromadb_retriever
from aatm.selectors import FirstResultSelector
from aatm.rerankers import Qwen3Reranker, BM25Reranker

model_name = "text-embedding-3-small"
reranker_name = "BM25"

translator = AsyncGeminiTranslator(model="gemini-2.5-flash")
retriever = load_chromadb_retriever(model_name)
selector = FirstResultSelector()

tm = TerminologyMapper(
    translator=translator,
    retriever=retriever,
    selector=selector, 
    reranker=BM25Reranker(),
    batch_size=10,
    rate_limit=model_registry[model_name].get("rate_limit"),
    output_dir=Path(model_registry[model_name]["output_path"])/"with-reranker"/reranker_name,
)

mapped_concepts_df = await tm.amap(
    'vocabularies/SOURCE_TO_CONCEPT_MAP.csv',
)